# Measles graph model

This notebook inspects the graph representation of UK regions and Local Authority upper-tier areas. Region nodes include measles-specific data where available; edges represent first- and second-order touching relationships.

In [ ]:
from pathlib import Path
import importlib
import sys

repo_root = Path.cwd()
if repo_root.name == "outbreak_risk_model":
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import outbreak_risk_model.notebook_helpers as notebook_helpers
importlib.reload(notebook_helpers)

build_and_show_geography_graph = notebook_helpers.build_and_show_geography_graph
load_measles_graph_tables = notebook_helpers.load_measles_graph_tables
node_measles_summary = notebook_helpers.node_measles_summary
node_neighbours = notebook_helpers.node_neighbours
show_geography_graph = notebook_helpers.show_geography_graph

## Whole UK region graph

In [ ]:
region_geojson = repo_root / "outbreak_risk_model/boundary_data/uk_regions_2025.geojson"

uk_region_geo_graph, uk_region_svg = build_and_show_geography_graph(
    region_geojson,
    title="Graph representation of UK regions",
    neighbours=4,
)

uk_region_svg

## Whole UK Local Authority upper-tier graph

In [ ]:
utla_geojson = repo_root / "outbreak_risk_model/boundary_data/uk_local_authority_upper_tier_2025.geojson"

utla_geo_graph, utla_svg = build_and_show_geography_graph(
    utla_geojson,
    title="Graph representation of local authorities (upper tier)",
    neighbours=4,
    max_label_count=0,
)

utla_svg

## Load measles graph-state tables

In [ ]:
region_graph = load_measles_graph_tables(prefix="measles_region_graph")
utla_graph = load_measles_graph_tables(prefix="measles_utla_graph")

In [ ]:
region_graph.nodes.head()

In [ ]:
region_graph.edges.head()

## Inspect one region node

In [ ]:
node_measles_summary(region_graph, "London")

In [ ]:
node_neighbours(region_graph, "London")

## Contact matrix used by graph nodes

In [ ]:
region_graph.contact_matrix

## Local Authority upper-tier topology

The UTLA graph currently contains topology only. Measles-specific LA-level population, vaccination, and case inputs still need to be added.

In [ ]:
utla_graph.nodes.head()

In [ ]:
utla_graph.edges.head()

## ODWP commuting-flow graph

These tables use Census 2021 ODWP usual-residence to workplace flows. Nodes are filtered to England-coded geographies; edges are directed from residence area to workplace area.

In [ ]:
region_odwp_graph = load_measles_graph_tables(prefix="measles_region_odwp_graph")
utla_odwp_graph = load_measles_graph_tables(prefix="measles_utla_odwp_graph")

len(region_odwp_graph.nodes), len(region_odwp_graph.edges), len(utla_odwp_graph.nodes), len(utla_odwp_graph.edges)

## England region ODWP graph

In [ ]:
show_geography_graph(
    region_odwp_graph,
    title="England regions: ODWP residence-to-workplace flows",
)

In [ ]:
region_odwp_graph.edges.loc[
    ~region_odwp_graph.edges["is_self_loop"],
    ["source_name", "target_name", "commuter_count", "mobility_weight", "destination_residence_share"],
].sort_values("commuter_count", ascending=False).head(20)

In [ ]:
def commuting_summary(edges):
    internal = (
        edges.loc[edges["is_self_loop"]]
        .set_index("source_name")["commuter_count"]
        .rename("internal_flow")
    )
    total_out = edges.groupby("source_name")["commuter_count"].sum().rename("total_outflow")
    total_in = edges.groupby("target_name")["commuter_count"].sum().rename("total_inflow")
    summary = total_out.to_frame().join(total_in, how="outer").join(internal, how="outer").fillna(0)
    for column in summary.columns:
        summary[column] = summary[column].astype(int)
    summary["external_outflow"] = summary["total_outflow"] - summary["internal_flow"]
    summary["external_inflow"] = summary["total_inflow"] - summary["internal_flow"]
    summary["net_external_inflow"] = summary["external_inflow"] - summary["external_outflow"]
    return summary.sort_values("net_external_inflow", ascending=False)

commuting_summary(region_odwp_graph.edges)

## London ODWP links

In [ ]:
region_odwp_graph.edges.loc[
    region_odwp_graph.edges["source_name"].eq("London") | region_odwp_graph.edges["target_name"].eq("London"),
    ["source_name", "target_name", "commuter_count", "mobility_weight", "destination_residence_share", "is_self_loop"],
].sort_values("commuter_count", ascending=False)

## Upper-tier local authority ODWP graph

In [ ]:
utla_odwp_graph.edges.loc[
    ~utla_odwp_graph.edges["is_self_loop"],
    ["source_name", "target_name", "commuter_count", "mobility_weight", "destination_residence_share"],
].sort_values("commuter_count", ascending=False).head(25)

In [ ]:
commuting_summary(utla_odwp_graph.edges).head(25)

## Network visualisations

This section regenerates project figures directly from the model inputs. In the regional figure, node **area** is proportional to population and directed arrows are based on the combined mobility matrix used by the outbreak model. The UTLA graph is an exploratory finer-resolution topology and was not used in the final forecasts.

Generated SVG, PNG and PDF files are written to `outbreak_risk_model/figures/`.

In [ ]:
from IPython.display import FileLink, Markdown, SVG, display
import outbreak_risk_model.network_figures as network_figure_module
importlib.reload(network_figure_module)
generate_all_formats = network_figure_module.generate_all_formats

network_figure_paths = generate_all_formats()

display(Markdown('### Regional population and mobility network'))
display(SVG(filename=str(network_figure_paths['england_population_mobility_network_svg'])))
display(Markdown('### Exploratory UK upper-tier local-authority topology'))
display(SVG(filename=str(network_figure_paths['uk_utla_exploratory_topology_svg'])))

display(Markdown('### Generated PDF files'))
for name, path in network_figure_paths.items():
    if name.endswith('_pdf'):
        display(FileLink(str(path)))
